<a href="https://colab.research.google.com/github/dahae0515/Baseline_Models/blob/main/craft_new_features.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Craft temporal features  

In [ ]:
import pandas as pd
from sklearn.feature_selection import VarianceThreshold
import numpy as np
import pickle

## ROSPaCe

In [ ]:
df = pd.read_csv('./data/datasets/rospace/bin_final_reduced_down0.3.csv')

/tmp/ipykernel_12450/2573016739.py:1: DtypeWarning: Columns (16,26,57,60,61) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('./data/datasets/rospace/bin_final_reduced_down0.3.csv')


In [ ]:
df.shape

(10081647, 63)

In [ ]:
label = df['attack']
df = df.drop(columns=['attack'])
timestamp = df['timestamp']
df = df.drop(columns=['timestamp'])
df = df.drop(columns=['Unnamed: 0']) # this is just a residual index

### Divide features
First numeric features:
- We divide them in Wireshark (5 features), OS (3 features) and ROS2 (2 features).
- We select Wireshark features semantically, meaning features that describe the evolution of the communication through time.
- OS features are selected eliminating too correlated features.
- ROS features have only 2 features and we keep them to be representative of the ROS layer.

#### Categorical

In [ ]:
list_column_cat = df.select_dtypes(exclude=[np.number]).columns

In [ ]:
list_column_cat

Index(['layers.sll.sll.unused', 'layers.ip.ip.flags', 'layers.ip.ip.checksum',
       'layers.tcp.tcp.flags_tree.tcp.flags.syn_tree._ws.expert._ws.expert.message',
       'layers.tcp.tcp.options', 'layers.tcp.tcp.payload', 'layers._ws.short',
       'src_topic', 'msg_type', 'msg_data'],
      dtype='object')

#### Numeric Features

In [ ]:
numeric_columns = df.select_dtypes(include=['number']).columns.tolist()
print(len(numeric_columns))

50


In [ ]:
numeric_columns

['layers.sll.sll.pkttype',
 'layers.sll.sll.hatype',
 'layers.ip.ip.version',
 'layers.ip.ip.dsfield_tree.ip.dsfield.ecn',
 'layers.ip.ip.flags_tree.ip.flags.rb',
 'layers.ip.ip.flags_tree.ip.flags.df',
 'layers.ip.ip.checksum.status',
 'layers.tcp.tcp.stream',
 'layers.tcp.tcp.ack',
 'layers.tcp.tcp.flags_tree.tcp.flags.ack',
 'layers.tcp.tcp.flags_tree.tcp.flags.syn',
 'layers.tcp.tcp.flags_tree.tcp.flags.syn_tree._ws.expert._ws.expert.severity',
 'layers.tcp.tcp.flags_tree.tcp.flags.syn_tree._ws.expert._ws.expert.group',
 'layers.tcp.tcp.window_size_value',
 'layers.tcp.tcp.window_size',
 'layers.tcp.tcp.options_tree.tcp.options.nop',
 'layers.tcp.tcp.options_tree.tcp.options.nop_tree.tcp.option_kind',
 'layers.tcp.tcp.analysis.tcp.analysis.acks_frame',
 'layers.tcp.tcp.analysis.tcp.analysis.initial_rtt',
 'layers.ssl.ssl.record.ssl.record.content_type',
 'layers.ipv6.ip.version',
 'layers.ipv6.ipv6.tclass_tree.ipv6.tclass.ecn',
 'layers.icmpv6.icmpv6.type',
 'Net_Sent',
 'pgpgin',


In [ ]:
ros_features = ['subscribers_count', 'publishers_count']
wire_features = ['layers.sll.sll.pkttype','layers.sll.sll.hatype','layers.ip.ip.version','layers.ip.ip.dsfield_tree.ip.dsfield.ecn',
                 'layers.ip.ip.flags_tree.ip.flags.rb','layers.ip.ip.flags_tree.ip.flags.df','layers.ip.ip.checksum.status','layers.tcp.tcp.stream',
                 'layers.tcp.tcp.ack','layers.tcp.tcp.flags_tree.tcp.flags.ack','layers.tcp.tcp.flags_tree.tcp.flags.syn','layers.tcp.tcp.flags_tree.tcp.flags.syn_tree._ws.expert._ws.expert.severity',
                 'layers.tcp.tcp.flags_tree.tcp.flags.syn_tree._ws.expert._ws.expert.group',
                 'layers.tcp.tcp.window_size_value','layers.tcp.tcp.window_size','layers.tcp.tcp.options_tree.tcp.options.nop','layers.tcp.tcp.options_tree.tcp.options.nop_tree.tcp.option_kind','layers.tcp.tcp.analysis.tcp.analysis.acks_frame',
                 'layers.tcp.tcp.analysis.tcp.analysis.initial_rtt','layers.ssl.ssl.record.ssl.record.content_type','layers.ipv6.ip.version','layers.ipv6.ipv6.tclass_tree.ipv6.tclass.ecn','layers.icmpv6.icmpv6.type']
os_features = [ 'Net_Sent','pgpgin','pgactivate','Disk_Read','pgfault','Net_Received','MemFree','Inactive','pgdeactivate','Tcp_Close','pgfree','nr_active_file','Cached',
                'nr_inactive_file','Disk_Write', 'pgpgout','Tcp_Syn','Buffers','Tcp_TimeWait','Tcp_Listen','Tcp_Established','Active','pgalloc_dma','pgmajfault','SwapFree',]

In [ ]:
len(ros_features), len(wire_features), len(os_features)

(2, 23, 25)

##### Numeric best for wireshark features

These are the features from wireshark on which make sense to compute mean, variance and slope.

In [ ]:
wire_features = ['layers.ip.ip.dsfield_tree.ip.dsfield.ecn', 'layers.tcp.tcp.ack', 'layers.tcp.tcp.window_size_value',
                 'layers.tcp.tcp.window_size', 'layers.tcp.tcp.analysis.tcp.analysis.initial_rtt']

##### Numeric best for OS features

###### Remove too higly correlated features

In [ ]:
import pandas as pd
import numpy as np

# Calculate correlation matrix
corr_matrix = df[os_features].corr()

# Select features with correlation > threshold (e.g., 0.95)
upper_triangle = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
to_drop_os = [column for column in upper_triangle.columns if any(upper_triangle[column] > 0.99)]

In [ ]:
len(to_drop_os)

2

In [ ]:
os_features = [item for item in os_features if item not in to_drop_os]

In [ ]:
len(os_features)

23

###### Rank using laplacian score

In [ ]:
# Initialize VarianceThreshold
selector = VarianceThreshold(threshold=0)  # Set threshold to 0 to calculate variance of all features
variances = df[os_features].var().values  # Calculate variance for each column
ranked_indices = np.argsort(variances)[::-1]
selected_features = df[os_features].columns[ranked_indices]

In [ ]:
selected_features = selected_features.to_list()

###### Selected best numerical features

In [ ]:
selected_num = selected_features[0:3] + wire_features + ros_features

In [ ]:
selected_num

['pgfault',
 'pgpgout',
 'pgpgin',
 'layers.ip.ip.dsfield_tree.ip.dsfield.ecn',
 'layers.tcp.tcp.ack',
 'layers.tcp.tcp.window_size_value',
 'layers.tcp.tcp.window_size',
 'layers.tcp.tcp.analysis.tcp.analysis.initial_rtt',
 'subscribers_count',
 'publishers_count']

#### All selected features

In [ ]:
list_column_cat = list_column_cat.to_list()

In [ ]:
all_features = list_column_cat + selected_num

In [ ]:
all_features.append('label')
all_features.append('timestamp')

In [ ]:
len(all_features)

22

In [ ]:
all_features

['layers.sll.sll.unused',
 'layers.ip.ip.flags',
 'layers.ip.ip.checksum',
 'layers.tcp.tcp.flags_tree.tcp.flags.syn_tree._ws.expert._ws.expert.message',
 'layers.tcp.tcp.options',
 'layers.tcp.tcp.payload',
 'layers._ws.short',
 'src_topic',
 'msg_type',
 'msg_data',
 'pgfault',
 'pgpgout',
 'pgpgin',
 'layers.ip.ip.dsfield_tree.ip.dsfield.ecn',
 'layers.tcp.tcp.ack',
 'layers.tcp.tcp.window_size_value',
 'layers.tcp.tcp.window_size',
 'layers.tcp.tcp.analysis.tcp.analysis.initial_rtt',
 'subscribers_count',
 'publishers_count',
 'label',
 'timestamp']

In [ ]:
df['label'] = label
df['timestamp'] = timestamp

In [ ]:
df.shape

(10081647, 62)

In [ ]:
df[all_features].to_csv('./data/datasets/rospace/rospace_selected_features.csv', index=False)

In [ ]:
df = pd.read_csv('./data/datasets/rospace/rospace_selected_features.csv')

/tmp/ipykernel_12450/3221965985.py:1: DtypeWarning: Columns (3,7,8,9) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('./data/datasets/rospace/rospace_selected_features.csv')


### Preprocess dataset

#### Remove NaN

In [ ]:
nanv = []
for col in df.columns:
    nanv.append(df[col].isnull().values.any())

In [ ]:
df.replace([np.inf, -np.inf], -1, inplace=True)
df.fillna(-1, inplace=True)
df = df.dropna(thresh=1, axis=1)

In [ ]:
nanv = []
for col in df.columns:
    nanv.append(df[col].isnull().values.any())

In [ ]:
df.to_csv('./data/datasets/rospace/rospace_selected_features_prepro.csv', index=False)

### Craft temporal Features  

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import linregress

In [ ]:
df = pd.read_csv('./data/datasets/rospace/rospace_selected_features_prepro.csv')

/tmp/ipykernel_10681/3677657908.py:1: DtypeWarning: Columns (3,7,8,9) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('./data/datasets/rospace/rospace_selected_features_prepro.csv')


In [ ]:
df

,layers.sll.sll.unused,layers.ip.ip.flags,layers.ip.ip.checksum,layers.tcp.tcp.flags_tree.tcp.flags.syn_tree._ws.expert._ws.expert.message,layers.tcp.tcp.options,layers.tcp.tcp.payload,layers._ws.short,src_topic,msg_type,msg_data,...,pgpgin,layers.ip.ip.dsfield_tree.ip.dsfield.ecn,layers.tcp.tcp.ack,layers.tcp.tcp.window_size_value,layers.tcp.tcp.window_size,layers.tcp.tcp.analysis.tcp.analysis.initial_rtt,subscribers_count,publishers_count,label,timestamp
0,08:0a,0x00004000,0x00001201,-1,01:01:08:0a:71:07:9c:74:30:28:11:5c,17:03:03:12:5f:50:97:d8:e0:97:9d:ac:ae:c8:c0:5...,[Packet size limited during capture: SSL trunc...,/mansarda1/image/image_raw/theora,Packet,<class 'theora_image_transport.msg._packet.Pac...,...,10528727.0,0.0,11385.0,501.0,501.0,-1.000000,1.0,1.0,0,2023-03-16 14:22:23.903192576
1,1d:67,0x00004000,0x0000819c,-1,01:01:08:0a:30:28:11:a0:71:07:9c:75,-1,-1,/mansarda1/image/image_raw/theora,Packet,<class 'theora_image_transport.msg._packet.Pac...,...,10528727.0,0.0,205554980.0,5054.0,5054.0,-1.000000,1.0,1.0,0,2023-03-16 14:22:23.904744448
2,01:06,0x00004000,0x00003cce,-1,-1,-1,-1,/mansarda1/image/image_raw/theora,Packet,<class 'theora_image_transport.msg._packet.Pac...,...,10528727.0,0.0,1.0,-1.0,0.0,0.000018,1.0,1.0,0,2023-03-16 14:22:23.905824512
3,08:0a,0x00004000,0x0000819a,-1,01:01:08:0a:30:28:11:a3:71:07:9c:78,-1,-1,/mansarda1/image/image_raw/theora,Packet,<class 'theora_image_transport.msg._packet.Pac...,...,10528727.0,0.0,205568636.0,5054.0,5054.0,-1.000000,1.0,1.0,0,2023-03-16 14:22:23.907192064
4,ca:bb,0x00004000,0x000007bf,-1,01:01:08:0a:71:07:9c:7a:30:28:11:a4,17:03:03:1c:88:24:69:99:18:ab:85:af:70:95:00:a...,[Packet size limited during capture: SSL trunc...,/mansarda1/image/image_raw/theora,Packet,<class 'theora_image_transport.msg._packet.Pac...,...,10528727.0,0.0,11385.0,501.0,501.0,-1.000000,1.0,1.0,0,2023-03-16 14:22:23.908826368
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10081642,04:06,0x00004000,0x0000f611,-1,01:01:08:0a:72:ab:58:2a:72:ab:57:ff,-1,-1,-1,-1,-1,...,86872729.0,0.0,44019810.0,24571.0,24571.0,-1.000000,-1.0,-1.0,1,2023-06-16 19:57:32.225117696
10081643,04:ac,0x00004000,0x0000da7f,-1,-1,-1,-1,-1,-1,-1,...,86872729.0,0.0,-1.0,-1.0,-1.0,-1.000000,-1.0,-1.0,1,2023-06-16 19:57:32.363449600
10081644,df:16,0x00004000,0x00005c61,-1,01:01:08:0a:72:ab:58:df:72:ab:58:df,-1,-1,-1,-1,-1,...,86872729.0,0.0,1.0,512.0,65536.0,0.000040,-1.0,-1.0,1,2023-06-16 19:57:32.405588992
10081645,68:00,0x00004000,0x00003dcb,-1,01:01:08:0a:72:ab:58:e0:72:ab:58:df,48:54:54:50:2f:31:2e:31:20:32:30:30:20:4f:4b:0...,[Packet size limited during capture: HTTP trun...,-1,-1,-1,...,86872729.0,0.0,105.0,512.0,65536.0,0.000040,-1.0,-1.0,1,2023-06-16 19:57:32.406351616


In [ ]:
label = df['label']

In [ ]:
df = df.drop(columns=['label'])

In [ ]:
numeric_columns = df.select_dtypes(include=['number']).columns.tolist()

In [ ]:
numeric_columns

['pgfault',
 'pgpgout',
 'pgpgin',
 'layers.ip.ip.dsfield_tree.ip.dsfield.ecn',
 'layers.tcp.tcp.ack',
 'layers.tcp.tcp.window_size_value',
 'layers.tcp.tcp.window_size',
 'layers.tcp.tcp.analysis.tcp.analysis.initial_rtt',
 'subscribers_count',
 'publishers_count']

In [ ]:
df['timestamp'] = pd.to_datetime(df['timestamp'], format='%Y-%m-%d %H:%M:%S.%f')

In [ ]:
def compute_slope(series):
    if len(series) > 1:
        x = np.arange(len(series))
        y = series.values
        slope, _, _, _, _ = linregress(x, y)
        return slope
    return np.nan

In [ ]:
time_windows = ['5S', '10S', '30S', '1min', '5min', '10min', '30min', '60min']
time_dfs = []

for time in time_windows:
    new_name = "time" + time
    df[new_name] = df["timestamp"].dt.floor(time)  # Create a time bin column

    df["row_count"] = df.groupby(new_name)["timestamp"].transform("count")  # Compute row count per time window

    agg_funcs = {col: ["mean", "var", compute_slope] for col in numeric_columns}  # Defines aggregation functions

    # Group by the time bin and compute aggregations
    grouped = df.groupby(new_name).agg(agg_funcs)

    # Flatten column names
    grouped.columns = [f"{col}_{stat}_{new_name}" for col, stat in grouped.columns]  # Flatten column names

    # Shift the index (timestamps) forward so that the first bin starts empty
    grouped = grouped.shift(1)  # This ensures first entry is empty after merging

    # Reset the index so 'new_name' becomes a regular column and not the index
    grouped.reset_index(inplace=True)

    time_dfs.append(grouped)
    print("Grouped DataFrame " + new_name + "\n", grouped.head())

Grouped DataFrame time5S
                time5S  pgfault_mean_time5S  pgfault_var_time5S  \
0 2023-03-16 14:22:20                  NaN                 NaN   
1 2023-03-16 14:22:25         2.512970e+09        1.046377e+09   
2 2023-03-16 14:22:30         2.513289e+09        2.258876e+10   
3 2023-03-16 14:22:35         2.513797e+09        2.273099e+10   
4 2023-03-16 14:22:40         2.514299e+09        2.038129e+10   

   pgfault_compute_slope_time5S  pgpgout_mean_time5S  pgpgout_var_time5S  \
0                           NaN                  NaN                 NaN   
1                    722.245692         6.605598e+07         4494.587196   
2                    799.238950         6.605654e+07       361904.168159   
3                    772.100643         6.605799e+07       326043.764412   
4                    751.007596         6.605936e+07       256213.792210   

   pgpgout_compute_slope_time5S  pgpgin_mean_time5S  pgpgin_var_time5S  \
0                           NaN               

In [ ]:
df.shape

(10081647, 30)

In [ ]:
time_dfs[0]

,time5S,pgfault_mean_time5S,pgfault_var_time5S,pgfault_compute_slope_time5S,pgpgout_mean_time5S,pgpgout_var_time5S,pgpgout_compute_slope_time5S,pgpgin_mean_time5S,pgpgin_var_time5S,pgpgin_compute_slope_time5S,...,layers.tcp.tcp.window_size_compute_slope_time5S,layers.tcp.tcp.analysis.tcp.analysis.initial_rtt_mean_time5S,layers.tcp.tcp.analysis.tcp.analysis.initial_rtt_var_time5S,layers.tcp.tcp.analysis.tcp.analysis.initial_rtt_compute_slope_time5S,subscribers_count_mean_time5S,subscribers_count_var_time5S,subscribers_count_compute_slope_time5S,publishers_count_mean_time5S,publishers_count_var_time5S,publishers_count_compute_slope_time5S
0,2023-03-16 14:22:20,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2023-03-16 14:22:25,2.512970e+09,1.046377e+09,722.245692,6.605598e+07,4.494587e+03,1.267271,10528727.0,0.0,0.0,...,1.505340,-0.960264,0.038412,-0.000174,1.152318,0.729978,0.006284,0.801325,0.360265,-0.001136
2,2023-03-16 14:22:30,2.513289e+09,2.258876e+10,799.238950,6.605654e+07,3.619042e+05,2.737757,10528727.0,0.0,0.0,...,0.387208,-0.973846,0.025510,0.000013,2.061538,6.822093,0.000606,3.570769,57.964938,0.006324
3,2023-03-16 14:22:35,2.513797e+09,2.273099e+10,772.100643,6.605799e+07,3.260438e+05,2.518856,10528727.0,0.0,0.0,...,-0.400861,-0.973333,0.025995,-0.000009,1.924444,9.479446,0.001305,2.657778,39.993986,0.003341
4,2023-03-16 14:22:40,2.514299e+09,2.038129e+10,751.007596,6.605936e+07,2.562138e+05,2.044877,10528727.0,0.0,0.0,...,-1.550470,-0.980213,0.019426,0.000022,1.888889,10.800136,0.000669,2.465753,37.099816,0.003265
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17469,2023-06-16 19:57:10,2.410085e+09,2.575959e+08,238.918357,8.975545e+07,5.747553e+06,33.784126,86872729.0,0.0,0.0,...,1452.916098,-0.461164,0.249655,0.000549,-0.442922,0.807507,-0.002708,-0.397260,0.983662,-0.003228
17470,2023-06-16 19:57:15,2.410120e+09,2.454780e+08,261.510413,8.975959e+07,4.125194e+05,11.134098,86872729.0,0.0,0.0,...,-420.047588,-0.451261,0.248922,-0.000157,-0.866667,0.250172,-0.001201,-0.851282,0.323130,-0.001385
17471,2023-06-16 19:57:20,2.410163e+09,1.994454e+08,125.758505,8.976267e+07,5.156377e+06,19.401134,86872729.0,0.0,0.0,...,-1014.992411,-0.390141,0.238653,0.000658,-0.739884,0.453883,-0.000634,-0.719653,0.544366,-0.000720
17472,2023-06-16 19:57:25,2.410202e+09,5.499434e+07,83.338929,8.976731e+07,3.269651e+05,6.372137,86872729.0,0.0,0.0,...,19.340689,-0.587389,0.243237,0.000495,-0.622378,0.614796,-0.003693,-0.604895,0.688958,-0.003897


In [ ]:
time_dfs[1]

,time10S,pgfault_mean_time10S,pgfault_var_time10S,pgfault_compute_slope_time10S,pgpgout_mean_time10S,pgpgout_var_time10S,pgpgout_compute_slope_time10S,pgpgin_mean_time10S,pgpgin_var_time10S,pgpgin_compute_slope_time10S,...,layers.tcp.tcp.window_size_compute_slope_time10S,layers.tcp.tcp.analysis.tcp.analysis.initial_rtt_mean_time10S,layers.tcp.tcp.analysis.tcp.analysis.initial_rtt_var_time10S,layers.tcp.tcp.analysis.tcp.analysis.initial_rtt_compute_slope_time10S,subscribers_count_mean_time10S,subscribers_count_var_time10S,subscribers_count_compute_slope_time10S,publishers_count_mean_time10S,publishers_count_var_time10S,publishers_count_compute_slope_time10S
0,2023-03-16 14:22:20,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2023-03-16 14:22:30,2.513229e+09,3.409168e+10,797.278594,6.605644e+07,3.425966e+05,2.114005,10528727.0,0.0,0.0,...,1.687898,-0.971285,0.027926,-0.000010,1.890137,5.797915,0.001408,3.048689,48.266376,0.006545
2,2023-03-16 14:22:40,2.514045e+09,8.452268e+10,755.498029,6.605867e+07,7.622646e+05,2.117975,10528727.0,0.0,0.0,...,-0.788841,-0.976726,0.022750,-0.000006,1.906907,10.123559,0.000210,2.563063,38.546734,0.000610
3,2023-03-16 14:22:50,2.515061e+09,8.645385e+10,727.572684,6.606187e+07,1.134364e+06,2.389710,10528727.0,0.0,0.0,...,-3.064303,-0.964259,0.034489,-0.000047,2.222302,15.373293,0.000951,3.155111,50.920845,0.001784
4,2023-03-16 14:23:00,2.515972e+09,7.819061e+10,1060.596540,6.606402e+07,1.214724e+06,3.728003,10528727.0,0.0,0.0,...,-0.726028,-0.975253,0.024163,0.000003,1.923510,10.446845,0.000684,2.503937,36.639906,0.003121
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9616,2023-06-16 19:56:50,2.409902e+09,8.311201e+08,192.830687,8.973825e+07,1.532157e+07,25.669293,86872729.0,0.0,0.0,...,263.161292,-0.474826,0.249892,0.000313,-0.674044,0.546764,-0.000639,-0.639839,0.694619,-0.000580
9617,2023-06-16 19:57:00,2.409978e+09,5.348633e+08,157.483950,8.974526e+07,1.073614e+06,7.134375,86872729.0,0.0,0.0,...,-756.114028,-0.493955,0.250487,0.000406,-0.405622,0.837151,0.001298,-0.373494,0.958813,0.001349
9618,2023-06-16 19:57:10,2.410063e+09,7.688666e+08,212.018037,8.975245e+07,1.504923e+07,28.980018,86872729.0,0.0,0.0,...,48.742292,-0.459254,0.248925,0.000086,-0.606335,0.633792,0.000979,-0.570136,0.785319,0.000958
9619,2023-06-16 19:57:20,2.410147e+09,6.424292e+08,155.052739,8.976156e+07,5.641495e+06,13.485620,86872729.0,0.0,0.0,...,-639.837955,-0.412171,0.242763,0.000321,-0.785582,0.383570,0.000102,-0.767098,0.467878,0.000083


In [ ]:
time_dfs[3].shape

(1902, 31)

In [ ]:
df['label'] = label

In [ ]:
for data, time in zip(time_dfs, time_windows):
    df_temp = df.merge(data, on='time' + str(time), how="left")
    print(df_temp.shape)
    print(df_temp)
    df_temp.to_csv('./data/datasets/rospace/aggregated/merged/rospace_merged_' + 'time' + str(time) + '.csv')

(10081647, 61)
         layers.sll.sll.unused layers.ip.ip.flags layers.ip.ip.checksum  \
0                        08:0a         0x00004000            0x00001201   
1                        1d:67         0x00004000            0x0000819c   
2                        01:06         0x00004000            0x00003cce   
3                        08:0a         0x00004000            0x0000819a   
4                        ca:bb         0x00004000            0x000007bf   
...                        ...                ...                   ...   
10081642                 04:06         0x00004000            0x0000f611   
10081643                 04:ac         0x00004000            0x0000da7f   
10081644                 df:16         0x00004000            0x00005c61   
10081645                 68:00         0x00004000            0x00003dcb   
10081646                 c5:64         0x00004000            0x0000404c   

         layers.tcp.tcp.flags_tree.tcp.flags.syn_tree._ws.expert._ws.expert.message 

In [ ]:
for data, time in zip(time_dfs, time_windows):
    data.to_csv('./data/datasets/rospace/aggregated/rospace_' + 'time' + str(time) + '.csv')

## Arancino

In [ ]:
df = pd.read_csv('./figshare_repo/all_experiments/data/arancino_attack_augmented_3.csv')

In [ ]:
df.head()

,_timestamp,cpu_load.load_15m,cpu_load.load_1m,cpu_load.load_5m,cpu_stats.ctx_switches,cpu_stats.interrupts,cpu_stats.soft_interrupts,cpu_times.idle,cpu_times.iowait,cpu_times.nice,...,vmstat.pgfault,vmstat.pgfree,vmstat.pginodesteal,vmstat.pgmajfault,vmstat.pgpgin,vmstat.pgpgout,vmstat.pgreuse,vmstat.pgrotated,vmstat.slabs_scanned,label
0,1,0.41,0.39,0.37,17956688.0,113118747.0,16015392.0,81214.56,11.15,1044.15,...,93479541.0,43580476.0,0.0,858.0,200230.0,56.0,40733614.0,34.0,0.0,0.0
1,2,0.41,0.39,0.37,17957193.0,113123609.0,16015848.0,81218.37,11.15,1044.19,...,93482292.0,43581643.0,0.0,858.0,200230.0,56.0,40735024.0,34.0,0.0,0.0
2,3,0.41,0.39,0.37,17957853.0,113127902.0,16016540.0,81221.68,11.15,1044.28,...,93488089.0,43585033.0,0.0,858.0,200230.0,56.0,40737252.0,34.0,0.0,0.0
3,4,0.41,0.39,0.37,17958439.0,113132516.0,16017095.0,81225.39,11.15,1044.30,...,93490877.0,43586205.0,0.0,858.0,200230.0,56.0,40738772.0,34.0,0.0,0.0
4,5,0.41,0.39,0.37,17959026.0,113137701.0,16017640.0,81229.15,11.15,1044.33,...,93493599.0,43587387.0,0.0,858.0,200230.0,56.0,40740200.0,34.0,0.0,0.0


In [ ]:
label = df['label']
df = df.drop(columns=['label'])

In [ ]:
df.shape

(207300, 120)

In [ ]:
list_column_string = df.select_dtypes(exclude=[np.number])

### Preprocess and feature selection

#### Remove NaN

In [ ]:
nanv = []
for col in df.columns:
    nanv.append(df[col].isnull().values.any())

In [ ]:
df.replace([np.inf, -np.inf], -1, inplace=True)
df.fillna(-1, inplace=True)
df = df.dropna(thresh=1, axis=1)

In [ ]:
nanv = []
for col in df.columns:
    nanv.append(df[col].isnull().values.any())

#### Remove features with only one value

In [ ]:
df = df.loc[:, df.nunique() > 1]

In [ ]:
df.shape

(207300, 120)

#### Remove too higly correlated features

In [ ]:
import pandas as pd
import numpy as np

# Calculate correlation matrix
corr_matrix = df.corr()
upper_triangle = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
to_drop = [column for column in upper_triangle.columns if any(upper_triangle[column] > 0.99)]

# Drop highly correlated features
df = df.drop(columns=to_drop)

In [ ]:
df.shape

(207300, 55)

#### Laplacian score to find representative features
We use an unsupervised method to be independent from the model.

In [ ]:
# Initialize VarianceThreshold
selector = VarianceThreshold(threshold=0)  # Set threshold to 0 to calculate variance of all features
variances = df.var().values  # Calculate variance for each column
ranked_indices = np.argsort(variances)[::-1]
selected_features = df.columns[ranked_indices]

In [ ]:
selected_features

Index(['disk.free', 'netinfo.wlan0.sent.bytes', 'netinfo.wlan0.rec.bytes',
       'cpu_stats.ctx_switches', 'redis_used_memory_rss',
       'meminfo.Committed_AS', 'redis_used_memory_peak', 'redis_used_memory',
       '_timestamp', 'netinfo.wlan0.rec.pkts', 'redis_used_memory_dataset',
       'redis_used_memory_overhead', 'netinfo.wlan0.sent.pkts',
       'meminfo.MemFree', 'meminfo.MemAvailable', 'meminfo.Cached',
       'meminfo.Active', 'meminfo.Inactive', 'meminfo.Shmem',
       'meminfo.Inactive(anon)', 'meminfo.Active(file)',
       'meminfo.Inactive(file)', 'meminfo.AnonPages', 'meminfo.Active(anon)',
       'meminfo.Buffers', 'vmstat.pgactivate', 'vmstat.nr_file_pages',
       'disk_io.busy_time', 'disk_io.read_time', 'temperature.temperature',
       'meminfo.Mapped', 'disk_io.read_merged_count', 'redis_active_keys',
       'meminfo.Slab', 'meminfo.KReclaimable', 'meminfo.SUnreclaim',
       'meminfo.PageTables', 'meminfo.KernelStack', 'vmstat.nr_kernel_stack',
       'vmstat.

In [ ]:
df['label'] = label

In [ ]:
df['label']

0         0.0
1         0.0
2         0.0
3         0.0
4         0.0
         ... 
207295    0.0
207296    0.0
207297    0.0
207298    0.0
207299    0.0
Name: label, Length: 207300, dtype: float64

In [ ]:
selected_features = selected_features[0:21].to_list()

In [ ]:
selected_features.append('label')

In [ ]:
df[selected_features]

,disk.free,netinfo.wlan0.sent.bytes,netinfo.wlan0.rec.bytes,cpu_stats.ctx_switches,redis_used_memory_rss,meminfo.Committed_AS,redis_used_memory_peak,redis_used_memory,_timestamp,netinfo.wlan0.rec.pkts,...,netinfo.wlan0.sent.pkts,meminfo.MemFree,meminfo.MemAvailable,meminfo.Cached,meminfo.Active,meminfo.Inactive,meminfo.Shmem,meminfo.Inactive(anon),meminfo.Active(file),label
0,8.448799e+09,66906535.0,32067248.0,17956688.0,7491584.0,781556.0,1823912.0,852276.0,1,141811.0,...,123782.0,501760.0,636100.0,181896.0,76444.0,306792.0,22184.0,213944.0,74516.0,0.0
1,8.448799e+09,66906837.0,32067669.0,17957193.0,7491584.0,781556.0,1823912.0,852276.0,2,141815.0,...,123783.0,501116.0,635456.0,181896.0,76448.0,306716.0,22184.0,213868.0,74516.0,0.0
2,8.448799e+09,66907139.0,32067811.0,17957853.0,7491584.0,781556.0,1823912.0,885044.0,3,141817.0,...,123784.0,501184.0,635524.0,181896.0,76444.0,307012.0,22184.0,214164.0,74516.0,0.0
3,8.448799e+09,66908135.0,32068864.0,17958439.0,7491584.0,781812.0,1823912.0,885044.0,4,141828.0,...,123790.0,501344.0,635684.0,181896.0,76452.0,307088.0,22184.0,214240.0,74516.0,0.0
4,8.448799e+09,66908437.0,32071860.0,17959026.0,7491584.0,781812.0,1823912.0,819508.0,5,141836.0,...,123791.0,501552.0,635892.0,181896.0,76448.0,307176.0,22184.0,214328.0,74516.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
207295,8.444289e+09,11294635.0,10272564.0,9599477.0,7548928.0,750792.0,1830924.0,886524.0,207296,77364.0,...,73211.0,475240.0,631532.0,190872.0,104392.0,305708.0,35344.0,207308.0,90624.0,0.0
207296,8.444289e+09,11295062.0,10273139.0,9600027.0,7548928.0,750792.0,1830924.0,886524.0,207297,77371.0,...,73213.0,475544.0,631836.0,190872.0,104392.0,305692.0,35344.0,207292.0,90624.0,0.0
207297,8.444289e+09,11295756.0,10273795.0,9600845.0,7548928.0,750792.0,1830924.0,886524.0,207298,77377.0,...,73218.0,476036.0,632328.0,190872.0,104392.0,305744.0,35344.0,207344.0,90624.0,0.0
207298,8.444289e+09,11297316.0,10275010.0,9602614.0,7548928.0,750792.0,1830924.0,886524.0,207299,77386.0,...,73228.0,475988.0,632280.0,190872.0,104392.0,305872.0,35344.0,207472.0,90624.0,0.0


#### Save the best 30 features

In [ ]:
df[selected_features].to_csv("./data/datasets/arancino/arancino_best_20.csv", index=False)

### Craft temporal Features  

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import linregress

In [ ]:
df = pd.read_csv("./data/datasets/arancino/arancino_best_20.csv")

In [ ]:
df

,disk.free,netinfo.wlan0.sent.bytes,netinfo.wlan0.rec.bytes,cpu_stats.ctx_switches,redis_used_memory_rss,meminfo.Committed_AS,redis_used_memory_peak,redis_used_memory,_timestamp,netinfo.wlan0.rec.pkts,...,netinfo.wlan0.sent.pkts,meminfo.MemFree,meminfo.MemAvailable,meminfo.Cached,meminfo.Active,meminfo.Inactive,meminfo.Shmem,meminfo.Inactive(anon),meminfo.Active(file),label
0,8.448799e+09,66906535.0,32067248.0,17956688.0,7491584.0,781556.0,1823912.0,852276.0,1,141811.0,...,123782.0,501760.0,636100.0,181896.0,76444.0,306792.0,22184.0,213944.0,74516.0,0.0
1,8.448799e+09,66906837.0,32067669.0,17957193.0,7491584.0,781556.0,1823912.0,852276.0,2,141815.0,...,123783.0,501116.0,635456.0,181896.0,76448.0,306716.0,22184.0,213868.0,74516.0,0.0
2,8.448799e+09,66907139.0,32067811.0,17957853.0,7491584.0,781556.0,1823912.0,885044.0,3,141817.0,...,123784.0,501184.0,635524.0,181896.0,76444.0,307012.0,22184.0,214164.0,74516.0,0.0
3,8.448799e+09,66908135.0,32068864.0,17958439.0,7491584.0,781812.0,1823912.0,885044.0,4,141828.0,...,123790.0,501344.0,635684.0,181896.0,76452.0,307088.0,22184.0,214240.0,74516.0,0.0
4,8.448799e+09,66908437.0,32071860.0,17959026.0,7491584.0,781812.0,1823912.0,819508.0,5,141836.0,...,123791.0,501552.0,635892.0,181896.0,76448.0,307176.0,22184.0,214328.0,74516.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
207295,8.444289e+09,11294635.0,10272564.0,9599477.0,7548928.0,750792.0,1830924.0,886524.0,207296,77364.0,...,73211.0,475240.0,631532.0,190872.0,104392.0,305708.0,35344.0,207308.0,90624.0,0.0
207296,8.444289e+09,11295062.0,10273139.0,9600027.0,7548928.0,750792.0,1830924.0,886524.0,207297,77371.0,...,73213.0,475544.0,631836.0,190872.0,104392.0,305692.0,35344.0,207292.0,90624.0,0.0
207297,8.444289e+09,11295756.0,10273795.0,9600845.0,7548928.0,750792.0,1830924.0,886524.0,207298,77377.0,...,73218.0,476036.0,632328.0,190872.0,104392.0,305744.0,35344.0,207344.0,90624.0,0.0
207298,8.444289e+09,11297316.0,10275010.0,9602614.0,7548928.0,750792.0,1830924.0,886524.0,207299,77386.0,...,73228.0,475988.0,632280.0,190872.0,104392.0,305872.0,35344.0,207472.0,90624.0,0.0


In [ ]:
label = df['label']

In [ ]:
df = df.drop(columns=['label'])

In [ ]:
timestamp = df['_timestamp']

In [ ]:
numeric_columns = df.select_dtypes(include=['number']).columns.tolist()

In [ ]:
numeric_columns = [col for col in numeric_columns if col != "_timestamp"]

In [ ]:
numeric_columns, len(numeric_columns)

(['disk.free',
  'netinfo.wlan0.sent.bytes',
  'netinfo.wlan0.rec.bytes',
  'cpu_stats.ctx_switches',
  'redis_used_memory_rss',
  'meminfo.Committed_AS',
  'redis_used_memory_peak',
  'redis_used_memory',
  'netinfo.wlan0.rec.pkts',
  'redis_used_memory_dataset',
  'redis_used_memory_overhead',
  'netinfo.wlan0.sent.pkts',
  'meminfo.MemFree',
  'meminfo.MemAvailable',
  'meminfo.Cached',
  'meminfo.Active',
  'meminfo.Inactive',
  'meminfo.Shmem',
  'meminfo.Inactive(anon)',
  'meminfo.Active(file)'],
 20)

In [ ]:
start_time = pd.Timestamp('2025-01-01 00:00:00')

# Add seconds to the start time and create a new column with the timestamps
df['_timestamp'] = start_time + pd.to_timedelta(df['_timestamp'], unit='s')

In [ ]:
def compute_slope(series):
    if len(series) > 1:
        x = np.arange(len(series))
        y = series.values
        slope, _, _, _, _ = linregress(x, y)
        return slope
    return np.nan

In [ ]:
time_windows = ['5S', '10S', '30S', '1min', '5min', '10min', '30min', '60min']
time_dfs = []

for time in time_windows:
    new_name = "time" + time
    df[new_name] = df["_timestamp"].dt.floor(time)  # Create a time bin column

    df["row_count"] = df.groupby(new_name)["_timestamp"].transform("count")  # Compute row count per time window

    agg_funcs = {col: ["mean", "var", compute_slope] for col in numeric_columns}  # Defines aggregation functions

    # Group by the time bin and compute aggregations
    grouped = df.groupby(new_name).agg(agg_funcs)

    # Flatten column names
    grouped.columns = [f"{col}_{stat}_{new_name}" for col, stat in grouped.columns]  # Flatten column names

    # Shift the index (timestamps) forward so that the first bin starts empty
    grouped = grouped.shift(1)  # This ensures first entry is empty after merging

    # Reset the index so 'new_name' becomes a regular column and not the index
    grouped.reset_index(inplace=True)

    time_dfs.append(grouped)
    print("Grouped DataFrame " + new_name + "\n", grouped.head())

Grouped DataFrame time5S
                time5S  disk.free_mean_time5S  disk.free_var_time5S  \
0 2025-01-01 00:00:00                    NaN                   NaN   
1 2025-01-01 00:00:05           8.448799e+09                   0.0   
2 2025-01-01 00:00:10           8.448799e+09                   0.0   
3 2025-01-01 00:00:15           8.448799e+09                   0.0   
4 2025-01-01 00:00:20           8.448799e+09                   0.0   

   disk.free_compute_slope_time5S  netinfo.wlan0.sent.bytes_mean_time5S  \
0                             NaN                                   NaN   
1                             0.0                            66907161.5   
2                             0.0                            66909457.4   
3                             0.0                            66912328.8   
4                             0.0                            66915024.0   

   netinfo.wlan0.sent.bytes_var_time5S  \
0                                  NaN   
1                 

In [ ]:
df.shape

(207300, 30)

In [ ]:
time_dfs[0]

,time5S,disk.free_mean_time5S,disk.free_var_time5S,disk.free_compute_slope_time5S,netinfo.wlan0.sent.bytes_mean_time5S,netinfo.wlan0.sent.bytes_var_time5S,netinfo.wlan0.sent.bytes_compute_slope_time5S,netinfo.wlan0.rec.bytes_mean_time5S,netinfo.wlan0.rec.bytes_var_time5S,netinfo.wlan0.rec.bytes_compute_slope_time5S,...,meminfo.Inactive_compute_slope_time5S,meminfo.Shmem_mean_time5S,meminfo.Shmem_var_time5S,meminfo.Shmem_compute_slope_time5S,meminfo.Inactive(anon)_mean_time5S,meminfo.Inactive(anon)_var_time5S,meminfo.Inactive(anon)_compute_slope_time5S,meminfo.Active(file)_mean_time5S,meminfo.Active(file)_var_time5S,meminfo.Active(file)_compute_slope_time5S
0,2025-01-01 00:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2025-01-01 00:00:05,8.448799e+09,0.0,0.0,66907161.5,4.820037e+05,510.2,32067898.0,4.718887e+05,499.0,...,118.4,22184.0,0.0,0.0,214054.0,3.113067e+04,118.4,74516.0,0.0,0.0
2,2025-01-01 00:00:10,8.448799e+09,0.0,0.0,66909457.4,6.868828e+05,510.2,32075460.2,6.025505e+06,1470.8,...,271.6,22184.0,0.0,0.0,214806.4,2.138608e+05,271.6,74516.0,0.0,0.0
3,2025-01-01 00:00:15,8.448799e+09,0.0,0.0,66912328.8,8.339957e+05,566.3,32081783.4,9.840596e+06,1769.1,...,138.0,22184.0,0.0,0.0,215707.2,5.084320e+04,138.0,74516.0,0.0,0.0
4,2025-01-01 00:00:20,8.448799e+09,0.0,0.0,66915024.0,8.880040e+05,579.6,32085375.2,3.272352e+05,348.0,...,163.2,22184.0,0.0,0.0,216599.2,8.019520e+04,163.2,74516.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
41456,2025-01-03 09:34:40,8.444289e+09,0.0,0.0,11276516.6,2.213363e+06,907.9,10259024.4,1.683346e+06,762.8,...,97.2,35372.0,0.0,0.0,213407.2,2.422720e+04,97.2,90624.0,0.0,0.0
41457,2025-01-03 09:34:45,8.444289e+09,0.0,0.0,11281240.2,3.750305e+06,1170.8,10262405.4,1.170485e+06,655.3,...,-1904.0,35360.8,235.2,-8.4,211220.0,1.176074e+07,-1904.0,90624.0,0.0,0.0
41458,2025-01-03 09:34:50,8.444289e+09,0.0,0.0,11286741.8,2.985135e+06,1064.8,10265795.2,1.577648e+06,744.3,...,42.8,35344.0,0.0,0.0,207763.2,9.891200e+03,42.8,90624.0,0.0,0.0
41459,2025-01-03 09:34:55,8.444289e+09,0.0,0.0,11291075.0,1.854862e+06,815.6,10269111.8,1.313783e+06,678.9,...,58.8,35344.0,0.0,0.0,207524.0,1.254400e+04,58.8,90624.0,0.0,0.0


In [ ]:
time_dfs[0].shape

(41461, 61)

In [ ]:
time_dfs[3].shape

(3456, 61)

In [ ]:
df['label'] = label

In [ ]:
df['label']

0         0.0
1         0.0
2         0.0
3         0.0
4         0.0
         ... 
207295    0.0
207296    0.0
207297    0.0
207298    0.0
207299    0.0
Name: label, Length: 207300, dtype: float64

In [ ]:
for data, time in zip(time_dfs, time_windows):
    df_temp = df.merge(data, on='time' + str(time), how="left")
    print(df_temp.shape)
    df_temp.to_csv('./data/datasets/arancino/aggregated/merged/arancino_merged' + 'time' + str(time) + ".csv")

(207300, 91)
(207300, 91)
(207300, 91)
(207300, 91)
(207300, 91)
(207300, 91)
(207300, 91)
(207300, 91)


In [ ]:
for data, time in zip(time_dfs, time_windows):
    data.to_csv('./data/datasets/arancino/aggregated/arancino_' + 'time' + str(time) + ".csv")

## DoS MQTT IoT  TODO

In [ ]:
df = pd.read_csv('./data/datasets/dos_mqtt_iot/train_set.csv')

ParserError: Error tokenizing data. C error: Calling read(nbytes) on source failed. Try engine='python'.

In [ ]:
df.shape

In [ ]:
label = df['label']
df = df.drop(columns=['label'])
timestamp = df['timestamp']
df = df.drop(columns=['timestamp'])
df = df.drop(columns=['Unnamed: 0']) # this is just a residual index

In [ ]:
df = df.drop(columns=['Unnamed: 0.1'])

In [ ]:
sequence = df['sequence']

In [ ]:
df = df.drop(columns=['sequence'])

### Divide features
First numeric features:
- We divide them in Wireshark (5 features), OS (3 features) and ROS2 (2 features).
- We select Wireshark features semantically, meaning features that describe the evolution of the communication through time.
- MQTT features are selected eliminating too correlated features.

#### Categorical

In [ ]:
list_column_cat = df.select_dtypes(exclude=[np.number]).columns

In [ ]:
list_column_cat

In [ ]:
cat_mqtt = ['layers.mqtt.mqtt.conack.flags',
       'layers.mqtt.mqtt.hdrflags', 'layers.mqtt.mqtt.msg',
       'layers.mqtt.mqtt.topic', 'layers.mqtt.mqtt.username']

In [ ]:
cat_wire = ['layers.arp.arp.src.proto_ipv4', 'layers.eth.eth.dst',
       'layers.eth.eth.dst_tree.eth.dst_resolved', 'layers.eth.eth.padding',
       'layers.eth.eth.src', 'layers.eth.eth.src_tree.eth.src_resolved',
       'layers.eth.eth.type', 'layers.frame.frame.protocols',
       'layers.ip.ip.addr', 'layers.ip.ip.checksum', 'layers.ip.ip.dsfield',
       'layers.ip.ip.flags', 'layers.ip.ip.id', 'layers.ip.ip.src',
       'layers.llc.llc.dsap',
       'layers.nbns.nbns.id', 'layers.stp.stp.protocol',
       'layers.tcp.tcp.analysis.tcp.analysis.flags._ws.expert._ws.expert.message',
       'layers.tcp.tcp.checksum', 'layers.tcp.tcp.flags',
       'layers.tcp.tcp.flags_tree.tcp.flags.fin_tree._ws.expert._ws.expert.message',
       'layers.tcp.tcp.flags_tree.tcp.flags.reset_tree._ws.expert._ws.expert.message',
       'layers.tcp.tcp.flags_tree.tcp.flags.str',
       'layers.tcp.tcp.flags_tree.tcp.flags.syn_tree._ws.expert._ws.expert.message',
       'layers.tcp.tcp.options',
       'layers.tcp.tcp.options_tree.tcp.options.sack',
       'layers.tcp.tcp.options_tree.tcp.options.wscale',
       'layers.tcp.tcp.payload', 'layers.tcp.tcp.segment_data',
       'layers.udp.udp.checksum']

#### Categorical best MQTT

In [ ]:
selected_features_cat= cat_wire[0:7] + cat_mqtt[0:3]

#### Numeric Features

In [ ]:
numeric_columns = df.select_dtypes(include=['number']).columns.tolist()
print(len(numeric_columns))

63


In [ ]:
numeric_columns

['layers.arp.arp.hw.type',
 'layers.arp.arp.opcode',
 'layers.eth.eth.dst_tree.eth.ig',
 'layers.eth.eth.dst_tree.eth.lg',
 'layers.eth.eth.len',
 'layers.frame.frame.len',
 'layers.frame.frame.number',
 'layers.ip.ip.dsfield_tree.ip.dsfield.dscp',
 'layers.ip.ip.len',
 'layers.ip.ip.proto',
 'layers.ip.ip.version',
 'layers.llc.llc.dsap_tree.llc.dsap.ig',
 'layers.llc.llc.oui',
 'layers.lldp.Chassis Subtype = MAC address, Id: 00:21:70:6d:99:d1.lldp.tlv.type',
 'layers.loop.loop.skipcount',
 'layers.mqtt.mqtt.connack.reason_code',
 'layers.mqtt.mqtt.hdrflags_tree.mqtt.dupflag',
 'layers.mqtt.mqtt.hdrflags_tree.mqtt.hdr_reserved',
 'layers.mqtt.mqtt.hdrflags_tree.mqtt.msgtype',
 'layers.mqtt.mqtt.hdrflags_tree.mqtt.qos',
 'layers.mqtt.mqtt.hdrflags_tree.mqtt.retain',
 'layers.mqtt.mqtt.msgid',
 'layers.mqtt.mqtt.passwd',
 'layers.mqtt.mqtt.proto_len',
 'layers.mqtt.mqtt.username_len',
 'layers.ssdp.http.prev_request_in',
 'layers.ssdp.http.request_number',
 'layers.tcp.tcp.ack',
 'layer

In [ ]:
mqtt_features = ['layers.mqtt.mqtt.msgid',
'layers.mqtt.mqtt.proto_len',
'layers.mqtt.mqtt.username_len' ]

wire_features = ['layers.frame.frame.len', 'layers.frame.frame.number', 'layers.eth.eth.len', 'layers.ip.ip.len',
'layers.ip.ip.dsfield_tree.ip.dsfield.dscp',
'layers.ip.ip.version',
'layers.tcp.tcp.ack',
'layers.tcp.tcp.analysis.tcp.analysis.ack_rtt',
'layers.tcp.tcp.analysis.tcp.analysis.bytes_in_flight',
'layers.tcp.tcp.analysis.tcp.analysis.duplicate_ack_num',
'layers.tcp.tcp.analysis.tcp.analysis.initial_rtt',
'layers.tcp.tcp.hdr_len',
'layers.tcp.tcp.len',
'layers.tcp.tcp.pdu.size',
'layers.tcp.tcp.seq',
'layers.tcp.tcp.stream',
'layers.tcp.tcp.window_size',
'layers.tcp.tcp.window_size_scalefactor',
'layers.tcp.tcp.window_size_value',
'layers.udp.udp.length',
'layers.udp.udp.stream']

In [ ]:
len(wire_features), len(mqtt_features)

(21, 3)

##### Numeric best for Wireshark features

###### Remove too higly correlated features

In [ ]:
import pandas as pd
import numpy as np

# Calculate correlation matrix
corr_matrix = df[wire_features].corr()

# Select features with correlation > threshold (e.g., 0.95)
upper_triangle = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
to_drop_wire = [column for column in upper_triangle.columns if any(upper_triangle[column] > 0.99)]

In [ ]:
len(to_drop_wire)

4

In [ ]:
wire_features = [item for item in wire_features if item not in to_drop_wire]

In [ ]:
len(wire_features)

17

###### Rank using laplacian score

In [ ]:
# Initialize VarianceThreshold
selector = VarianceThreshold(threshold=0)  # Set threshold to 0 to calculate variance of all features
variances = df[wire_features].var().values  # Calculate variance for each column
ranked_indices = np.argsort(variances)[::-1]
selected_features_wire = df[wire_features].columns[ranked_indices]

In [ ]:
selected_features_wire = selected_features_wire.to_list()

##### Numeric best for MQTT features

###### Remove too higly correlated features

In [ ]:
import pandas as pd
import numpy as np

# Calculate correlation matrix
corr_matrix = df[mqtt_features].corr()

# Select features with correlation > threshold (e.g., 0.95)
upper_triangle = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
to_drop_mqtt = [column for column in upper_triangle.columns if any(upper_triangle[column] > 0.99)]

In [ ]:
len(to_drop_mqtt)

0

###### Rank using laplacian score

In [ ]:
# Initialize VarianceThreshold
selector = VarianceThreshold(threshold=0)  # Set threshold to 0 to calculate variance of all features
variances = df[mqtt_features].var().values  # Calculate variance for each column
ranked_indices = np.argsort(variances)[::-1]
selected_features_mqtt = df[mqtt_features].columns[ranked_indices]

In [ ]:
selected_features_mqtt = selected_features_mqtt.to_list()

###### Selected best numerical features

In [ ]:
selected_num = selected_features_wire[0:7] + selected_features_mqtt

In [ ]:
selected_num

['layers.tcp.tcp.ack',
 'layers.tcp.tcp.seq',
 'layers.frame.frame.number',
 'layers.tcp.tcp.window_size',
 'layers.tcp.tcp.stream',
 'layers.tcp.tcp.window_size_value',
 'layers.tcp.tcp.analysis.tcp.analysis.bytes_in_flight',
 'layers.mqtt.mqtt.msgid',
 'layers.mqtt.mqtt.username_len',
 'layers.mqtt.mqtt.proto_len']

#### All selected features

In [ ]:
all_features = selected_features_cat + selected_num

In [ ]:
len(all_features)

20

In [ ]:
df['label'] = label

In [ ]:
all_features.append('label')

In [ ]:
df['timestamp'] = timestamp

In [ ]:
all_features.append('timestamp')

In [ ]:
df['sequence'] = sequence

In [ ]:
all_features.append('sequence')

In [ ]:
all

In [ ]:
df[all_features].to_csv('./data/datasets/dos_mqtt_iot/dos_mqtt_iot_selected_features_train.csv', index=False)

In [ ]:
df = pd.read_csv('./data/datasets/dos_mqtt_iot/test_set.csv')

/tmp/ipykernel_8608/3083775639.py:1: DtypeWarning: Columns (3,47,79,87,94) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('./data/datasets/dos_mqtt_iot/test_set.csv')


In [ ]:
df_train = pd.read_csv('./data/datasets/dos_mqtt_iot/dos_mqtt_iot_selected_features_train.csv')

/tmp/ipykernel_8608/1917596725.py:1: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df_train = pd.read_csv('./data/datasets/dos_mqtt_iot/dos_mqtt_iot_selected_features_train.csv')


In [ ]:
df.shape

(3191295, 102)

In [ ]:
df[df_train.columns].to_csv('./data/datasets/dos_mqtt_iot/dos_mqtt_iot_selected_features_test.csv', index=False)

### Preprocess dataset

In [ ]:
df = pd.read_csv('./data/datasets/dos_mqtt_iot/dos_mqtt_iot_selected_features_test.csv')

/tmp/ipykernel_8608/2585128626.py:1: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('./data/datasets/dos_mqtt_iot/dos_mqtt_iot_selected_features_test.csv')


In [ ]:
df.shape

(3191295, 23)

#### Remove NaN

In [ ]:
nanv = []
for col in df.columns:
    nanv.append(df[col].isnull().values.any())

In [ ]:
df.replace([np.inf, -np.inf], -1, inplace=True)
df.fillna(-1, inplace=True)
df = df.dropna(thresh=1, axis=1)

In [ ]:
nanv = []
for col in df.columns:
    nanv.append(df[col].isnull().values.any())

In [ ]:
df.to_csv('./data/datasets/dos_mqtt_iot/dos_mqtt_iot_selected_features_prepro.csv', index=False)

### Craft temporal Features  

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import linregress

In [ ]:
df = pd.read_csv('./data/datasets/dos_mqtt_iot/dos_mqtt_iot_selected_features_prepro.csv')

/tmp/ipykernel_8608/1437420675.py:1: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('./data/datasets/dos_mqtt_iot/dos_mqtt_iot_selected_features_prepro.csv')


In [ ]:
df

,layers.arp.arp.src.proto_ipv4,layers.eth.eth.dst,layers.eth.eth.dst_tree.eth.dst_resolved,layers.eth.eth.padding,layers.eth.eth.src,layers.eth.eth.src_tree.eth.src_resolved,layers.eth.eth.type,layers.mqtt.mqtt.conack.flags,layers.mqtt.mqtt.hdrflags,layers.mqtt.mqtt.msg,...,layers.tcp.tcp.window_size,layers.tcp.tcp.stream,layers.tcp.tcp.window_size_value,layers.tcp.tcp.analysis.tcp.analysis.bytes_in_flight,layers.mqtt.mqtt.msgid,layers.mqtt.mqtt.username_len,layers.mqtt.mqtt.proto_len,label,timestamp,sequence
0,-1,00:21:70:6d:99:d1,Dell_6d:99:d1,-1,00:09:b7:6f:ad:70,Cisco_6f:ad:70,0x00000800,-1,-1,-1,...,64240.0,1.0,64240.0,-1.0,-1.0,-1.0,-1.0,normal,2021-06-06 07:31:08.767976,-1.0
1,-1,00:09:b7:6f:ad:70,Cisco_6f:ad:70,-1,00:21:70:6d:99:d1,Dell_6d:99:d1,0x00000800,-1,-1,-1,...,65535.0,1.0,65535.0,-1.0,-1.0,-1.0,-1.0,normal,2021-06-06 07:31:08.768123,-1.0
2,-1,00:21:70:6d:99:d1,Dell_6d:99:d1,00:00:00:00:00:00,00:09:b7:6f:ad:70,Cisco_6f:ad:70,0x00000800,-1,-1,-1,...,64256.0,1.0,502.0,-1.0,-1.0,-1.0,-1.0,normal,2021-06-06 07:31:08.771240,-1.0
3,-1,00:21:70:6d:99:d1,Dell_6d:99:d1,-1,00:09:b7:6f:ad:70,Cisco_6f:ad:70,0x00000800,-1,0x00000010,-1,...,64256.0,1.0,502.0,37.0,-1.0,4.0,4.0,normal,2021-06-06 07:31:08.772414,-1.0
4,-1,00:09:b7:6f:ad:70,Cisco_6f:ad:70,-1,00:21:70:6d:99:d1,Dell_6d:99:d1,0x00000800,-1,-1,-1,...,513.0,2.0,513.0,-1.0,-1.0,-1.0,-1.0,normal,2021-06-06 07:31:08.772512,-1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3191290,-1,00:21:70:6d:99:d1,Dell_6d:99:d1,00:00:00:00:00:00,00:09:b7:6f:ad:70,Cisco_6f:ad:70,0x00000800,-1,-1,-1,...,0.0,1321.0,0.0,-1.0,-1.0,-1.0,-1.0,WILL_DDoS_AD,2021-06-06 12:48:27.513075,49.5
3191291,-1,00:21:70:6d:99:d1,Dell_6d:99:d1,00:00:00:00:00:00,00:09:b7:6f:ad:70,Cisco_6f:ad:70,0x00000800,-1,-1,-1,...,64256.0,20000.0,502.0,-1.0,-1.0,-1.0,-1.0,normal,2021-06-06 12:48:27.544607,-1.0
3191292,-1,00:09:b7:6f:ad:70,Cisco_6f:ad:70,-1,00:21:70:6d:99:d1,Dell_6d:99:d1,0x00000800,-1,-1,-1,...,130816.0,20000.0,511.0,-1.0,-1.0,-1.0,-1.0,normal,2021-06-06 12:48:27.544645,-1.0
3191293,-1,00:21:70:6d:99:d1,Dell_6d:99:d1,00:00:00:00:00:00,00:09:b7:6f:ad:70,Cisco_6f:ad:70,0x00000800,-1,-1,-1,...,64256.0,1322.0,502.0,-1.0,-1.0,-1.0,-1.0,WILL_DDoS_AD,2021-06-06 12:48:27.546071,49.5


In [ ]:
label = df['label']

In [ ]:
df = df.drop(columns=['label'])

In [ ]:
timestamp = df['timestamp']

In [ ]:
sequence = df['sequence']

In [ ]:
df = df.drop(columns=['sequence'])

In [ ]:
numeric_columns = df.select_dtypes(include=['number']).columns.tolist()

In [ ]:
numeric_columns = [col for col in numeric_columns if col != "timestamp"]

In [ ]:
numeric_columns, len(numeric_columns)

(['layers.tcp.tcp.ack',
  'layers.tcp.tcp.seq',
  'layers.frame.frame.number',
  'layers.tcp.tcp.window_size',
  'layers.tcp.tcp.stream',
  'layers.tcp.tcp.window_size_value',
  'layers.tcp.tcp.analysis.tcp.analysis.bytes_in_flight',
  'layers.mqtt.mqtt.msgid',
  'layers.mqtt.mqtt.username_len',
  'layers.mqtt.mqtt.proto_len'],
 10)

In [ ]:
timestamp

0          2021-06-06 07:31:08.767976
1          2021-06-06 07:31:08.768123
2          2021-06-06 07:31:08.771240
3          2021-06-06 07:31:08.772414
4          2021-06-06 07:31:08.772512
                      ...            
3191290    2021-06-06 12:48:27.513075
3191291    2021-06-06 12:48:27.544607
3191292    2021-06-06 12:48:27.544645
3191293    2021-06-06 12:48:27.546071
3191294    2021-06-06 12:48:27.546144
Name: timestamp, Length: 3191295, dtype: object

In [ ]:
df['timestamp'] = pd.to_datetime(df['timestamp'], format='%Y-%m-%d %H:%M:%S.%f')

In [ ]:
df['timestamp']

0         2021-06-06 07:31:08.767976
1         2021-06-06 07:31:08.768123
2         2021-06-06 07:31:08.771240
3         2021-06-06 07:31:08.772414
4         2021-06-06 07:31:08.772512
                     ...            
3191290   2021-06-06 12:48:27.513075
3191291   2021-06-06 12:48:27.544607
3191292   2021-06-06 12:48:27.544645
3191293   2021-06-06 12:48:27.546071
3191294   2021-06-06 12:48:27.546144
Name: timestamp, Length: 3191295, dtype: datetime64[ns]

In [ ]:
df['label'] = label

In [ ]:
df['sequence'] = sequence

In [ ]:
def compute_slope(series):
    if len(series) > 1:
        x = np.arange(len(series))
        y = series.values
        slope, _, _, _, _ = linregress(x, y)
        return slope
    return np.nan

In [ ]:
time_windows = ['5S', '10S', '30S', '1min', '5min', '10min', '30min', '60min']
time_dfs = []

for time in time_windows:
    new_name = "time" + time
    df[new_name] = df["timestamp"].dt.floor(time)  # Create a time bin column

    df["row_count"] = df.groupby(new_name)["timestamp"].transform("count")  # Compute row count per time window

    agg_funcs = {col: ["mean", "var", compute_slope] for col in numeric_columns}  # Defines aggregation functions

    # Group by the time bin and compute aggregations
    grouped = df.groupby(new_name).agg(agg_funcs)

    # Flatten column names
    grouped.columns = [f"{col}_{stat}_{new_name}" for col, stat in grouped.columns]  # Flatten column names

    # Shift the index (timestamps) forward so that the first bin starts empty
    grouped = grouped.shift(1)  # This ensures first entry is empty after merging

    # Reset the index so 'new_name' becomes a regular column and not the index
    grouped.reset_index(inplace=True)

    time_dfs.append(grouped)
    print("Grouped DataFrame " + new_name + "\n", grouped.head())

Grouped DataFrame time5S
                time5S  layers.tcp.tcp.ack_mean_time5S  \
0 2021-06-06 07:31:05                             NaN   
1 2021-06-06 07:31:10                       29.906832   
2 2021-06-06 07:31:15                       55.996909   
3 2021-06-06 07:31:20                       88.009119   
4 2021-06-06 07:31:25                      115.736655   

   layers.tcp.tcp.ack_var_time5S  layers.tcp.tcp.ack_compute_slope_time5S  \
0                            NaN                                      NaN   
1                    2811.235016                                 0.208497   
2                   14280.414851                                 0.074645   
3                   53545.251058                                 0.100619   
4                  111564.044608                                -0.113405   

   layers.tcp.tcp.seq_mean_time5S  layers.tcp.tcp.seq_var_time5S  \
0                             NaN                            NaN   
1                       25.40372

In [ ]:
df.shape

(3191295, 32)

In [ ]:
time_dfs[0]

,time5S,layers.tcp.tcp.ack_mean_time5S,layers.tcp.tcp.ack_var_time5S,layers.tcp.tcp.ack_compute_slope_time5S,layers.tcp.tcp.seq_mean_time5S,layers.tcp.tcp.seq_var_time5S,layers.tcp.tcp.seq_compute_slope_time5S,layers.frame.frame.number_mean_time5S,layers.frame.frame.number_var_time5S,layers.frame.frame.number_compute_slope_time5S,...,layers.tcp.tcp.analysis.tcp.analysis.bytes_in_flight_compute_slope_time5S,layers.mqtt.mqtt.msgid_mean_time5S,layers.mqtt.mqtt.msgid_var_time5S,layers.mqtt.mqtt.msgid_compute_slope_time5S,layers.mqtt.mqtt.username_len_mean_time5S,layers.mqtt.mqtt.username_len_var_time5S,layers.mqtt.mqtt.username_len_compute_slope_time5S,layers.mqtt.mqtt.proto_len_mean_time5S,layers.mqtt.mqtt.proto_len_var_time5S,layers.mqtt.mqtt.proto_len_compute_slope_time5S
0,2021-06-06 07:31:05,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2021-06-06 07:31:10,29.906832,2.811235e+03,0.208497,25.403727,2.561180e+03,0.245692,82.000000,2.173500e+03,0.999997,...,0.010950,6723.795031,2.454643e+08,175.379865,-0.397516,4.065994,-0.002922,-0.565217,1.997283,-0.002444
2,2021-06-06 07:31:15,55.996909,1.428041e+04,0.074645,52.245750,1.476459e+04,0.082370,486.000000,3.493800e+04,1.000000,...,-0.001278,7123.987635,2.771380e+08,7.367951,-0.412674,4.032223,-0.000060,-0.582689,1.915366,-0.000016
3,2021-06-06 07:31:20,88.009119,5.354525e+04,0.100619,90.495441,6.003413e+04,0.107525,1138.500000,3.613517e+04,1.000000,...,0.001811,7254.831307,2.788980e+08,13.013122,-0.414894,4.051346,-0.000129,-0.589666,1.886164,-0.000063
4,2021-06-06 07:31:25,115.736655,1.115640e+05,-0.113405,123.558719,1.277892e+05,-0.119251,1748.500000,2.636717e+04,1.000000,...,-0.009372,6192.110320,2.265921e+08,-1.227892,-0.405694,4.209451,0.000213,-0.590747,1.882125,0.000143
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3804,2021-06-06 12:48:05,15191.093178,2.939330e+09,21.949890,17814.439268,3.509763e+09,22.084289,240103.000000,3.015017e+04,1.000000,...,0.000192,5047.920133,1.724462e+08,-1.528038,-0.405990,4.208231,-0.000306,-0.592346,1.875208,-0.000193
3805,2021-06-06 12:48:10,14583.796358,2.713211e+09,17.045445,17249.485099,3.269971e+09,17.298150,240705.500000,3.045167e+04,1.000000,...,0.004662,4764.468543,1.672077e+08,1.819911,-0.427152,3.966492,-0.000330,-0.594371,1.866701,-0.000210
3806,2021-06-06 12:48:15,9495.585770,1.885294e+09,-9.603444,11447.334308,2.331716e+09,-10.887434,142948.679337,1.353005e+10,-240.749946,...,0.093211,3137.752437,1.186513e+08,-3.633941,-0.393762,4.135532,-0.000100,-0.576023,1.942020,-0.000044
3807,2021-06-06 12:48:20,2271.807842,3.694608e+08,0.585480,2707.810536,4.511776e+08,0.706160,46616.625561,7.713983e+09,-0.497394,...,0.027329,1968.643221,8.885652e+07,-0.694833,-0.433104,3.753982,-0.000010,-0.597426,1.851359,-0.000007


In [ ]:
time_dfs[0].shape

(3809, 31)

In [ ]:
time_dfs[3].shape

(318, 31)

In [ ]:
for data, time in zip(time_dfs, time_windows):
    df_temp = df.merge(data, on='time' + str(time), how="left")
    print(df_temp.shape)
    df_temp.to_csv('./data/datasets/dos_mqtt_iot/aggregated/merged/dos_mqtt_iot_merged_' + 'time' + str(time) + "_test.csv")

(3191295, 62)
(3191295, 62)
(3191295, 62)
(3191295, 62)
(3191295, 62)
(3191295, 62)
(3191295, 62)
(3191295, 62)


In [ ]:
for data, time in zip(time_dfs, time_windows):
    data.to_csv('./data/datasets/dos_mqtt_iot/aggregated/dos_mqtt_iot_' + 'time' + str(time) + "_test.csv")